# IND320 project work, part 1

Viktor, NMBU, autumn 2026.

- GitHub repository: https://github.com/viktorbgulbrandsen/ind320-viktorbgulbrandsen
- Streamlit app: https://ind320-viktorbgulbrandsen.streamlit.app/
- Video: https://nmbu.cloud.panopto.eu/Panopto/Pages/Viewer.aspx?id=86b9fb43-7fb7-41d1-bce4-b4d401091d75

In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

## Reading the data

In [7]:
#read with pandas
df = pd.read_csv("../data/reservoirs.csv", parse_dates=["dato_Id"])

print(df.shape)
df.head()

(14877, 11)


,dato_Id,omrType,omrnr,iso_aar,iso_uke,fyllingsgrad,kapasitet_TWh,fylling_TWh,neste_Publiseringsdato,fyllingsgrad_forrige_uke,endring_fyllingsgrad
0,1995-09-03,EL,4,1995,35,0.944721,21.079208,19.913979,0001-01-01T00:00:00,0.936888,0.007833
1,2020-11-15,EL,1,2020,46,0.974361,6.003264,5.849344,2020-11-25T13:00:00,0.997406,-0.023046
2,2011-08-28,EL,4,2011,34,0.786499,21.079208,16.578772,0001-01-01T00:00:00,0.787193,-0.000694
3,1998-07-05,EL,3,1998,27,0.793969,8.920932,7.082947,0001-01-01T00:00:00,0.733812,0.060157
4,2011-03-27,EL,4,2011,12,0.300820,21.079208,6.341054,0001-01-01T00:00:00,0.311122,-0.010301


The Pandas print shows the file we are analyzing to have 14 877 rows over 11 columns. Note that in order to get this print, we prefix the path with ".." because the relative path is given from "notebooks/" and the file we read is in "data/".

The Pandas argument 'parse_dates' reads 'dato_Id' as dates rather than text, so that it can serve as the time axis. First and foremost, we notice that the rows as now printed are in random order (1995, 2020, ...).

In [8]:
areas = df.groupby(["omrType", "omrnr"])
areas.size()

omrType  omrnr
EL       1        1653
         2        1653
         3        1653
         4        1653
         5        1653
NO       0        1653
VASS     1        1653
         2        1653
         3        1653
dtype: int64

To understand why we have more rows than we have weeks, we use `groupby` on `omrType` and `omrnr`. This shows that the data gives multiple rows with the same date, one per area, where the national total covers all the areas together. We, however, only care about the national total.

## Sorting and renaming

In [9]:
#translate to english, keep the norwegian names as keys
columns = {
    "dato_Id": "date",
    "omrType": "area_type",
    "omrnr": "area_no",
    "iso_aar": "iso_year",
    "iso_uke": "iso_week",
    "fyllingsgrad": "fill_level",
    "kapasitet_TWh": "capacity_TWh",
    "fylling_TWh": "fill_TWh",
    "fyllingsgrad_forrige_uke": "fill_level_prev_week",
    "endring_fyllingsgrad": "fill_level_change",
}

#share of rows with the placeholder date
placeholder = df["neste_Publiseringsdato"] == "0001-01-01T00:00:00"
print(placeholder.mean())

#drop publiseringsdato
df = df.drop(columns="neste_Publiseringsdato")
df = df.rename(columns=columns)
#sort by date, fix index
df = df.sort_values("date")
df = df.reset_index(drop=True)
df.head()

0.7610405323653963


,date,area_type,area_no,iso_year,iso_week,fill_level,capacity_TWh,fill_TWh,fill_level_prev_week,fill_level_change
0,1995-01-08,EL,5,1995,1,0.616448,17.390587,10.720388,0.614288,0.002160
1,1995-01-08,EL,4,1995,1,0.558864,21.079208,11.780413,0.801348,-0.242484
2,1995-01-08,EL,3,1995,1,0.602376,8.920932,5.373754,0.596027,0.006349
3,1995-01-08,EL,1,1995,1,0.621467,6.003264,3.730831,0.734888,-0.113421
4,1995-01-08,EL,2,1995,1,0.630947,34.043590,21.479713,0.777604,-0.146657


I create a map from the Norwegian headings to English. 'neste_Publiseringsdato' is just when NVE publishes its next report, and holds a placeholder '0001-01-01' in 76% of the rows. It says nothing about the reservoirs relevant to us, and is therefore dropped.

# Log

**16 september**

After initalizing with uv, reading the requirements with 'uv add -r requirements.txt'. I commit and so on. 

To get Jupyter to work, I add

uv add --dev jupyter

**17 september**

Renaming 'test.py' to '1_home.py'. The home page now says what the app is about and where the data comes from.

'streamlit_app.py' is now only the menu. A list of pages and 'st.navigation'. Every new page is one more line in the list.

**17 september**

Making the table page. One row per column, with a small line chart of the first month using 'st.column_config.LineChartColumn'.

The data is in long format. The same columns hold 9 areas, so "each column" does not mean anything before I pick one. I use the national total ('NO') for now.

I first scaled every column to 0 to 1 so they would fit the same chart. Then it turns out each cell scales to its own min and max by default, so I drop the scaling.

**20 september**

Making the plot page. A selectbox for one column or all, and a select_slider over months that starts on the first month.

The data is weekly, so I turn the dates into months with 'to_period("M")' and use those as the slider options.

Putting all columns in one plot does not work at first. 'fill_level' is between 0 and 1 and the TWh columns go up to about 87, so the small ones just lie flat at the bottom. I put the TWh columns on a second y-axis on the right.

**21 september**

Making page 4, the free page. It shows how full the reservoirs are in each area in the last week of the data. This is the first page that uses all 9 areas and not only the national total. The area codes are not readable, so I turn them into names like 'NO1' and 'Water region 2'.

Looking back, the hardest part was the data, not Streamlit. The long format with 9 areas in the same columns, and the very different scales, took more thinking than the app itself. I also drop 'neste_Publiseringsdato', since it is '0001-01-01' in most rows.

The most interesting part was how little code a Streamlit page needs. A title, a table or a plot, and it runs in the browser.

**22 september**

Filling in the notebook. Reading the CSV, sorting by date and renaming the columns to English with a dictionary, so the Norwegian names are kept as keys.



# AI Log

** 16 september** 

Claude told me to also ignore '__pycache__/' in my git history. This is, naturally, a good recommendation.

I prefer using 'uv' over 'pip' for better package handling. Claude recommends me to add to README.md that Jupter is in a dev group. 

**17 september**

Claude told me that with 'st.navigation', Streamlit only shows the pages in the list, and no longer picks up everything in 'pages/' by itself.

Claude checked the Streamlit source and found that 'LineChartColumn' scales each cell by itself, so my own scaling was not needed.

**20 september**

I asked Claude how to get two y-axes in plotly. It showed me 'make_subplots' with 'secondary_y', and I use that for the TWh columns.

**21 september**

Claude explained the area codes to me. 'EL' 1 to 5 are the price areas NO1 to NO5, 'VASS' 1 to 3 are water regions, and 'NO' is the whole country.

**22 september**

Claude fixed spelling and small errors in my notebook text, like 'parse_dates' being an argument and not a function.
